### Libs

In [5]:
import requests
import json
import pandas as pd
from time import sleep
from datetime import datetime, timedelta
from IPython.display import HTML, display
import time
import re
import csv

# Google Sheets
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### API CRM

In [2]:
#### QUERY

def query_appointments(current_page,start_date,end_date):

  token = "145418|arQc09gsrcSNJipgDRaM4Ep6rl3aJGkLtDMnxa0u"
  endpoint = "https://open-api.eprocorpo.com.br/graphql"
  headers = {'Content-Type': 'application/json',
             "authorization": f"Bearer {token}"}

  query = """query ($filters: AppointmentFiltersInput, $pagination: PaginationInput) {
                    fetchAppointments(filters: $filters, pagination: $pagination) {
                        meta {
                            lastPage
                        }
                        data {
                            id
                            status {
                                label
                            }
                            employee {
                                name
                            }
                            store {
                                name
                            }
                            customer {
                                id
                                name
                                telephones {
                                    number
                                }
                            }
                            procedure{
                                name
                            }
                            startDate
                        }
                    }
                }"""

  variables = {
                "filters": {
                    "startDateRange": {
                        "start": start_date,
                        "end": end_date,
                    },
                },
                "pagination": {
                    "currentPage": current_page,
                    "perPage": 1000,
                },
            }

  response = requests.post(endpoint, json={"query": query,"variables":variables}, headers=headers)

  if response.status_code == 200:

    response = response.json()
    total_pages = response["data"]["fetchAppointments"]["meta"]["lastPage"]

    print(f"\rQuerying - {current_page}/{total_pages} pages",end="")

    progress_index = current_page//(total_pages/100)
    out.update(progress(progress_index, 100))

    return response

  else:
    print(response)
    return "error"



def progress(value, max=100):
    return HTML("""
        <progress
            value='{value}'
            max='{max}',
            style='width: 80%'
        >
            {value}
        </progress>
    """.format(value=value, max=max))

###### FETCH
days_to_offset = 5 # Seleciona a quantidade de dias antes e depois
d_minus_30 = (datetime.today() - timedelta(days=days_to_offset)).strftime('%Y-%m-%d')
d_plus_30 = (datetime.today() + timedelta(days=days_to_offset)).strftime('%Y-%m-%d')

def create_appointmens(file_name,start_date,end_date):
  # Faz as requisições e salva o resultado em um arquivo CSV
  folder_path = "/content/drive/MyDrive/_whatsapp-project/appointments_30d"

  save_path = f"{folder_path}/{file_name}"

  current_page = 1

  api_response = query_appointments(current_page,start_date,end_date)

  if (api_response == "error"):
    print("Something went wrong with the query")
    print("Waiting a bit...")
    sleep(10)
    print("Trying Again!")
    api_response = query_appointments(current_page,start_date,end_date)

    if api_response == "error":
      print("Error with the query... aborting")

  api_data = api_response["data"]["fetchAppointments"]["data"]
  api_data_length = len(api_data)


  results_list = [["appointments_id","status_label", "employee_name",
                   "store_name","customer_id","customer_name","telephones","procedure_name","startDate"]]

  while (api_data_length > 0):

    for data_row in api_data:

      appointments_id = data_row["id"]

      status_data = data_row["status"]
      status_label = status_data["label"]

      try:
        store_data = data_row["store"]
        store_name = store_data["name"]
      except:
        store_name = ""

      customer_data = data_row["customer"]
      customer_name = customer_data["name"]
      customer_id = customer_data["id"]
      employee_name = data_row["employee"]["name"]

      try:
        customer_telephones = customer_data["telephones"][0]
        telephone = customer_telephones["number"]
      except:
        telephone = ""


      procedure_data = data_row["procedure"]
      procedure_name = procedure_data["name"]

      startDate = data_row["startDate"]

      results_row = [appointments_id,status_label,employee_name, store_name,customer_id,customer_name,telephone,procedure_name,startDate]

      results_list.append(results_row)

    current_page += 1

    api_response = query_appointments(current_page,start_date,end_date)

    if (api_response == "error"):
      print("Something went wrong with the query")
      print("Waiting a bit...")
      sleep(10)
      print("Trying Again!")
      api_response = query_appointments(current_page,start_date,end_date)

      if api_response == "error":
        print("Error with the query... aborting")
        break

    api_data = api_response["data"]["fetchAppointments"]["data"]
    api_data_length = len(api_data)

  df = pd.DataFrame(results_list[1:],columns = results_list[0])
  df.to_csv(save_path, index=False)
  print('\r' + ' ' * 30, end='')
  print('\r', end='')
  print(f"Query Concluida com sucesso! - {start_date} - {end_date}")

out = display(progress(0, 100), display_id=True)
create_appointmens("appointments_5d.csv",d_minus_30,d_plus_30)

Query Concluida com sucesso! - 2024-11-16 - 2024-11-26


# Plástica

### SocialHub

In [3]:
# Envias as mensagens por meio da API do Social Hub
# 11988444797: B2TwOhCJSwO8VWYmEtoL1o7MX2PqPz1p

url = "https://apinew.socialhub.pro/api/sendMessage"
api_token = 'B2TwOhCJSwO8VWYmEtoL1o7MX2PqPz1p'

def send_message(telephone, message):

    telephone = str(telephone)
    request_data = {
        "api_token": api_token,  # ID da API no Social HUB BOTOX
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(request_data))

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def send_message_with_file(telephone, message,file_path):
    url = "https://apinew.socialhub.pro/api/sendMessage"

    telephone = str(telephone)

    request_data = {
        "api_token": api_token,
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, data=request_data, files=files)

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def enviar_mensagem(telefone,mensagem,file_name):

  if file_name == None:
    data = send_message(telefone,mensagem)
  else:
    data = send_message_with_file(telefone,mensagem,file_name)

  return data

### Load Databases

In [4]:
# Função para atualizar CSV de Mensagens

def update_csv_mensagens(dados_mensagens):
  with open(mensagens_log_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile, delimiter=',')
    writer.writerow(dados_mensagens)

In [5]:
# Carregando bases

appointents_file = "/content/drive/MyDrive/_whatsapp-project/appointments_30d/appointments_5d.csv"
mensagens_log_file = "/content/drive/MyDrive/_whatsapp-project/sent_messages_confirmation_pl.csv"

appointents_df = pd.read_csv(appointents_file) # Relatório de agendamentos do CRM
base_mensagens_df = pd.read_csv(mensagens_log_file) # Log de mensagens enviadas

In [6]:
# Save base_mensagens_df to csv url
base_mensagens_df

,telefone,status_fluxo,nome_da_mensagem,data_de_envio
0,11963546222,Agendado,agd_d2,2024-10-09
1,11963546222,Agendado,agd_d2,2024-10-09
2,11963546222,Agendado,agd_d2,2024-10-10
3,11963546222,Agendado,agd_d1,2024-10-10
4,11963546222,Agendado,agd_d0,2024-10-10
...,...,...,...,...
1619,12997524979,Agendado,agd_d2,2024-11-19
1620,13988700876,Agendado,agd_d0,2024-11-19
1621,16981087681,Agendado,agd_d2,2024-11-19
1622,18996742286,Agendado,agd_d0,2024-11-19


In [7]:
# create new column data_de_envio
base_mensagens_df["data_de_envio"] = datetime.today().strftime('%Y-%m-%d')
# Filtra log de mensagens excluindo mensagens enviadas há mais de 30 dias
base_mensagens_df['data_de_envio'] = pd.to_datetime(base_mensagens_df['data_de_envio'])
date_30_days_ago = datetime.now() - timedelta(days=30)
base_mensagens_df = base_mensagens_df[base_mensagens_df['data_de_envio'] >= date_30_days_ago]

# Groupby do log de mensagens
contar_mensagens = base_mensagens_df.groupby(["telefone","status_fluxo"]).agg({"telefone":"count"})
contar_mensagens = contar_mensagens.rename(columns={"telefone": "mensagens_count"})

contar_mensagens = contar_mensagens.reset_index()
contar_mensagens["telefone"] = contar_mensagens["telefone"].astype(str)

In [8]:
import re

# Função para limpar telefones
def clean_telephone(telefone):
    telefone = str(telefone).split('.')[0]  # Remove qualquer coisa após o ponto decimal
    telefone = re.sub(r'[^\d]', '', telefone)  # Remove todos os caracteres não numéricos
    if telefone.startswith('55'):
        telefone = telefone[2:]  # Remove o prefixo +55
    return telefone

# Arruma os telefones
appointents_df["telephones"] = appointents_df["telephones"].astype(str)
appointents_df["telephones"] = appointents_df["telephones"].apply(clean_telephone)

In [9]:
# Pega o texto e outros dados das mensagens

url_da_tabela_de_mensagens = 'https://docs.google.com/spreadsheets/d/1iNt8-2qmQKYD0McDDrUUb9QLBn5G2h6M5yUa8oo2isg/edit'
ss = gc.open_by_url(url_da_tabela_de_mensagens)
sheet = ss.worksheet("Confirma_PL")
mensagens = sheet.get_all_records()

# Transforma os dados em um dicionário para hash indexing
mensagens_dic = {}
for item in mensagens:
  tipo_de_mensagem = item["nome_mensagem"]
  mensagens_dic[tipo_de_mensagem] = item

### Creating filtered df

In [10]:
# Cria a coluna com a diferença de datas
appointents_df['startDate'] = pd.to_datetime(appointents_df['startDate'])
appointents_df['startDate'] = appointents_df['startDate'].dt.normalize()
today = datetime.today()
appointents_df['dias_do_agendamento'] = (today - appointents_df['startDate']).dt.days

agendado_regex = r".*Agendado.*|.*Reagendado.*|.*Atendido.*"
unidade_regex_inclusiva = "PLÁSTICA" # Adicionar unidades que queremos ver no relatório

# Lista de procedimentos a serem excluídos
procedimentos_exclusao = [
    'SEGUNDA OPINIÃO (AVALIAÇÃO CIRURGIA)',
    'RETORNO DE AVALIAÇÃO (ENTREGA EXAMES)',
    'RETORNO CIRURGIA PLÁSTICA'
]

# Atualiza a máscara para filtrar procedimentos válidos excluindo os indesejados
procedimento_mask = ~appointents_df['procedure_name'].isin(procedimentos_exclusao) # Exclui procedimentos indesejados
unidade_incluir_mask = appointents_df['store_name'].str.contains(unidade_regex_inclusiva, na=False, case=False) # Filtra somente unidades selecionadas

# Cria coluna flag de agendamento
appointents_df["eh_agendamento"] =  appointents_df["status_label"].str.contains(agendado_regex, na=False, case=False)

# Filtra base de agendamentos
filtered_df = appointents_df.loc[procedimento_mask & unidade_incluir_mask]

# Acrescenta coluna com a quantidade de mensagens por status (cancelado, falta)
filtered_df = filtered_df.merge(contar_mensagens, how='left', left_on=["telephones","status_label"], right_on=["telefone","status_fluxo"])
filtered_df.drop(columns=["telefone","status_fluxo"],inplace=True)
filtered_df.fillna(0,inplace=True)

# Adiciona coluna com a flag que diz se o cliente tem algum agendamento
filtered_df["tem agendamento"] = filtered_df.groupby(by=["telephones"])["eh_agendamento"].transform("max")

# Cria base telefones únicos para enviar mensagem
df_para_mandar_mensagens = filtered_df.groupby(by=["telephones","status_label"]).agg({"dias_do_agendamento":"min",'mensagens_count':'max',"store_name":"first","customer_name":"first", "employee_name":"first","startDate":"max"}).reset_index()

status_atendido_regex = r".*Atendido.*|.*Falta.*|.*Cancelado.*"
status_atendido_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_atendido_regex, na=False, case=False) # Remove Atendio e Confirmado
status_reagendado = 'Reagendado'
status_reagendado_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_reagendado, na=False, case=False) # Remove Reagendado

df_para_mandar_mensagens = df_para_mandar_mensagens.loc[status_atendido_mask & status_reagendado_mask]

# Pega telefones únicos com a data mais recente
df_para_mandar_mensagens = df_para_mandar_mensagens.loc[df_para_mandar_mensagens.groupby('telephones')['dias_do_agendamento'].idxmin()]

In [11]:
df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate
1,11910349212,Confirmado,0,0.0,PLÁSTICA,Chirleide Sales da Silva,Rebeca Lopes Granja,2024-11-21
2,11910471911,Confirmado,0,0.0,PLÁSTICA,Ariana Da Fonseca Faturi,Rebeca Lopes Granja,2024-11-21
3,11910661113,Agendado,0,1.0,PLÁSTICA,Mirian Nonato de Paula,Rebeca Lopes Granja,2024-11-21
5,11911171345,Agendado,-1,0.0,PLÁSTICA,Carleide De Souza Barbosa Costa,Rebeca Lopes Granja,2024-11-22
6,11912233947,Agendado,-1,0.0,PLÁSTICA,Adriana Cristina Nery Borges,Rebeca Lopes Granja,2024-11-22
...,...,...,...,...,...,...,...,...
253,16981087681,Agendado,0,1.0,PLÁSTICA,Flavia Pereira Neves,Lídia Traboulsi,2024-11-21
256,18998267305,Agendado,-4,0.0,PLÁSTICA,Joscileia Figueira Andrade,Osvaldo Ribeiro de Almeida Neto,2024-11-25
258,4396434758,Agendado,-4,0.0,PLÁSTICA,Alexandre Antônio Luiz,Osvaldo Ribeiro de Almeida Neto,2024-11-25
259,63993111053,Agendado,0,2.0,PLÁSTICA,Gilvanny Sousa Oliveira,Rebeca Lopes Granja,2024-11-21


In [12]:
# Final treatment
df_para_mandar_mensagens['mensagens_count'] = df_para_mandar_mensagens['mensagens_count'].astype(int)
df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate
1,11910349212,Confirmado,0,0,PLÁSTICA,Chirleide Sales da Silva,Rebeca Lopes Granja,2024-11-21
2,11910471911,Confirmado,0,0,PLÁSTICA,Ariana Da Fonseca Faturi,Rebeca Lopes Granja,2024-11-21
3,11910661113,Agendado,0,1,PLÁSTICA,Mirian Nonato de Paula,Rebeca Lopes Granja,2024-11-21
5,11911171345,Agendado,-1,0,PLÁSTICA,Carleide De Souza Barbosa Costa,Rebeca Lopes Granja,2024-11-22
6,11912233947,Agendado,-1,0,PLÁSTICA,Adriana Cristina Nery Borges,Rebeca Lopes Granja,2024-11-22
...,...,...,...,...,...,...,...,...
253,16981087681,Agendado,0,1,PLÁSTICA,Flavia Pereira Neves,Lídia Traboulsi,2024-11-21
256,18998267305,Agendado,-4,0,PLÁSTICA,Joscileia Figueira Andrade,Osvaldo Ribeiro de Almeida Neto,2024-11-25
258,4396434758,Agendado,-4,0,PLÁSTICA,Alexandre Antônio Luiz,Osvaldo Ribeiro de Almeida Neto,2024-11-25
259,63993111053,Agendado,0,2,PLÁSTICA,Gilvanny Sousa Oliveira,Rebeca Lopes Granja,2024-11-21


In [13]:
# # Criando o DataFrame com uma linha de dados personalizados
# data = {
#     'telephones': ['11963546222'],  # Coloque seu número aqui no formato correto
#     'status_label': ['Confirmado'],  # Personalize o status
#     'dias_do_agendamento': [0],  # Dias até o agendamento
#     'mensagens_count': [0],  # Contagem de mensagens
#     'store_name': ['TATUAPÉ'],  # Nome da loja
#     'customer_name': ['Luis Faria'],  # Nome do cliente
#     'employee_name': ['Prestador Teste'],  # Nome do funcionário
#     'startDate': ['2024-08-14']  # Data do início
# }

# df_para_mandar_mensagens = pd.DataFrame(data)
# df_para_mandar_mensagens

### Send Message

In [14]:
from datetime import datetime

def ajusta_mensagem(row, texto_mensagem):
    store_name = row["store_name"].capitalize()
    customer_name = row["customer_name"].split()[0].capitalize()
    prestador_name = row["employee_name"].split()[0].capitalize()

    # Convert startDate to the desired format if it's a Timestamp or datetime object
    startDate = row["startDate"]
    if isinstance(startDate, datetime):
        startDate = startDate.strftime('%d/%m/%Y')
    else:
        startDate = datetime.strptime(str(startDate), '%Y-%m-%d').strftime('%d/%m/%Y')

    texto_mensagem = texto_mensagem.replace("[nome]", customer_name)
    texto_mensagem = texto_mensagem.replace("[prestador]", prestador_name)
    texto_mensagem = texto_mensagem.replace("[data]", startDate)

    return texto_mensagem

In [15]:
# import time

# # Total time to sleep
# total_time = 1 * 60 * 60  #+ (15 * 60) # 1 hour + 15m
# interval = 60
# display_interval = 1  # countdown
# # Loop that sleeps until time to go
# for remaining_time in range(int(total_time), 0, -interval):

#     time.sleep(interval)
#     if remaining_time % (display_interval * 60) == 0:
#         remaining_minutes = remaining_time // 60
#         print(f"Time left: {remaining_minutes} minutes")

# print("Finished waiting.. ready for next")

In [ ]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-2
  if status_label == "Agendado" and dias_do_agendamento == -2:# and mensagens_count == 0:
    nome_mensagem = "agd_d2"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Agendado d-1
  if status_label == "Agendado" and dias_do_agendamento == -1:# and mensagens_count == 0:
    nome_mensagem = "agd_d1"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Agendado d-0
  if status_label == "Agendado" and dias_do_agendamento == 0:# and mensagens_count == 0:
    nome_mensagem = "agd_d0"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-1
  if status_label == "Confirmado" and dias_do_agendamento >= 1:# and mensagens_count == 0:
    nome_mensagem = "conf_d1"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-0
  if status_label == "Confirmado" and dias_do_agendamento == 0:# and mensagens_count == 0:
    nome_mensagem = "conf_d0"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

conf_d0 enviada com sucesso para: 11910349212
Resting 10s...
conf_d0 enviada com sucesso para: 11910471911
Resting 10s...
agd_d0 enviada com sucesso para: 11910661113
Resting 10s...
agd_d1 enviada com sucesso para: 11911171345
Resting 10s...
agd_d1 enviada com sucesso para: 11912233947
Resting 10s...
agd_d1 enviada com sucesso para: 11914076660
Resting 10s...
agd_d0 enviada com sucesso para: 11914862566
Resting 10s...
agd_d0 enviada com sucesso para: 11930046742
Resting 10s...
conf_d0 enviada com sucesso para: 11934686036
Resting 10s...
conf_d0 enviada com sucesso para: 11940180104
Resting 10s...
conf_d0 enviada com sucesso para: 11940314065
Resting 10s...
conf_d0 enviada com sucesso para: 11940393462
Resting 10s...
agd_d1 enviada com sucesso para: 11942093199
Resting 10s...
conf_d0 enviada com sucesso para: 11945041081
Resting 10s...
agd_d0 enviada com sucesso para: 11945925719
Resting 10s...
agd_d1 enviada com sucesso para: 11946284178
Resting 10s...
agd_d0 enviada com sucesso para: 

_______________

# HOMA

### Social Hub

In [ ]:
# Envias as mensagens por meio da API do Social Hub
# (55) 11943062144: TzLDeLL16ZPv3iKNBbYEgSLmYIC3nq7x

url = "https://apinew.socialhub.pro/api/sendMessage"
api_token = 'TzLDeLL16ZPv3iKNBbYEgSLmYIC3nq7x'

def send_message(telephone, message):

    telephone = str(telephone)
    request_data = {
        "api_token": api_token,  # ID da API no Social HUB BOTOX
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(request_data))

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def send_message_with_file(telephone, message,file_path):
    url = "https://apinew.socialhub.pro/api/sendMessage"

    telephone = str(telephone)

    request_data = {
        "api_token": api_token,
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, data=request_data, files=files)

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def enviar_mensagem(telefone,mensagem,file_name):

  if file_name == None:
    data = send_message(telefone,mensagem)
  else:
    data = send_message_with_file(telefone,mensagem,file_name)

  return data

### Load Databases

In [ ]:
# Função para atualizar CSV de Mensagens

def update_csv_mensagens_hsr(dados_mensagens):
  with open(mensagens_log_file_hsr, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile, delimiter=',')
    writer.writerow(dados_mensagens)

In [ ]:
# Carregando bases

appointents_file = "/content/drive/MyDrive/_whatsapp-project/appointments_30d/appointments_5d.csv"
mensagens_log_file_hsr = "/content/drive/MyDrive/_whatsapp-project/sent_messages_confirmation_hsr.csv"

appointents_df = pd.read_csv(appointents_file) # Relatório de agendamentos do CRM
base_mensagens_df_hsr = pd.read_csv(mensagens_log_file_hsr) # Log de mensagens enviadas

In [ ]:
base_mensagens_df_hsr

,telefone,status_fluxo,nome_da_mensagem,data_de_envio
0,11963546222,Agendado,agd_d1,10/24/24
1,11963546222,Agendado,agd_d1,2024-11-04
2,1120430356,Confirmado,conf_d1,2024-11-04
3,11913384277,Agendado,agd_d1,2024-11-04
4,11913467796,Agendado,agd_d1,2024-11-04
...,...,...,...,...
538,11998274824,Agendado,agd_d0,2024-11-11
539,11998325538,Confirmado,conf_d1,2024-11-11
540,11998857909,Confirmado,conf_d1,2024-11-11
541,11999699608,Agendado,agd_d0,2024-11-11


In [ ]:
import re

# Função para limpar telefones
def clean_telephone(telefone):
    telefone = str(telefone).split('.')[0]  # Remove qualquer coisa após o ponto decimal
    telefone = re.sub(r'[^\d]', '', telefone)  # Remove todos os caracteres não numéricos
    if telefone.startswith('55'):
        telefone = telefone[2:]  # Remove o prefixo +55
    return telefone

# Arruma os telefones
appointents_df["telephones"] = appointents_df["telephones"].astype(str)
appointents_df["telephones"] = appointents_df["telephones"].apply(clean_telephone)

In [ ]:
# create new column data_de_envio
base_mensagens_df_hsr["data_de_envio"] = datetime.today().strftime('%Y-%m-%d')
# Filtra log de mensagens excluindo mensagens enviadas há mais de 30 dias
base_mensagens_df_hsr['data_de_envio'] = pd.to_datetime(base_mensagens_df_hsr['data_de_envio'])
date_30_days_ago = datetime.now() - timedelta(days=30)
base_mensagens_df_hsr = base_mensagens_df_hsr[base_mensagens_df_hsr['data_de_envio'] >= date_30_days_ago]

# Groupby do log de mensagens
contar_mensagens = base_mensagens_df_hsr.groupby(["telefone","status_fluxo"]).agg({"telefone":"count"})
contar_mensagens = contar_mensagens.rename(columns={"telefone": "mensagens_count"})

contar_mensagens = contar_mensagens.reset_index()
contar_mensagens["telefone"] = contar_mensagens["telefone"].astype(str)

In [ ]:
# Pega o texto e outros dados das mensagens

url_da_tabela_de_mensagens = 'https://docs.google.com/spreadsheets/d/1iNt8-2qmQKYD0McDDrUUb9QLBn5G2h6M5yUa8oo2isg/edit'
ss = gc.open_by_url(url_da_tabela_de_mensagens)
sheet = ss.worksheet("Confirma_HSR")
mensagens = sheet.get_all_records()

# Transforma os dados em um dicionário para hash indexing
mensagens_dic = {}
for item in mensagens:
  tipo_de_mensagem = item["nome_mensagem"]
  mensagens_dic[tipo_de_mensagem] = item

In [ ]:
mensagens_dic

{'agd_d4': {'nome_mensagem': 'agd_d4',
  'texto_mensagem': 'Olá, [nome] aqui é a Rafaela do Hospital São Rafael, tudo bem? ☺️\n\nVocê tem uma avaliação agendada para dia [data] às [hora] no Ambulatório do Hospital. \nPodemos confirmar?\n1- Confirmar \n2- Reagendar\n\nSegue nosso endereço:\n📍Rua Eça de Queiroz, 488 - Vila Mariana \n(Próximo ao metrô Paraíso)\n⚠️Lembrando que o valor da avaliação é de R$ 80,00 mas caso feche o contrato o \nmesmo é abatido \n⚠️Esperamos que você tenha uma excelente experiência �',
  'file_id': '',
  'descrição_mensagem': ''},
 'agd_d1': {'nome_mensagem': 'agd_d1',
  'texto_mensagem': 'Bom dia! Tudo bem?\n\nSomos do Hospital São Rafael ☺️\nEstá chegando o dia da sua avaliação é amanhã dia [data] às *[hora]*.\n\n⚠️ Não esqueça seu documento com foto e chegar com 15min de antecedência do horário agendado.\nEstamos te aguardando!',
  'file_id': '',
  'descrição_mensagem': ''},
 'conf_d1': {'nome_mensagem': 'conf_d1',
  'texto_mensagem': 'Bom dia! Tudo bem?\n\

### Data Prep

In [ ]:
# Converte a coluna 'startDate' para o formato datetime
appointents_df['startDate'] = pd.to_datetime(appointents_df['startDate'])

# Cria uma nova coluna 'hora_agendamento' com apenas a hora do agendamento
appointents_df['hora_agendamento'] = appointents_df['startDate'].dt.time

# Normaliza a data (remove a hora da coluna 'startDate')
appointents_df['startDate'] = appointents_df['startDate'].dt.normalize()

# Calcula a diferença de dias
today = datetime.today()
appointents_df['dias_do_agendamento'] = (today - appointents_df['startDate']).dt.days

In [ ]:
agendado_regex = r".*Agendado.*|.*Reagendado.*|.*Atendido.*"
unidade_regex_inclusiva = "HOMA" # Adicionar unidades que queremos ver no relatório

# Lista de procedimentos a serem excluídos
procedimentos_exclusao = [
    'SEGUNDA OPINIÃO (AVALIAÇÃO CIRURGIA)',
    'RETORNO DE AVALIAÇÃO (ENTREGA EXAMES)',
    'RETORNO CIRURGIA PLÁSTICA'
]

# Atualiza a máscara para filtrar procedimentos válidos excluindo os indesejados
procedimento_mask = ~appointents_df['procedure_name'].isin(procedimentos_exclusao) # Exclui procedimentos indesejados
unidade_incluir_mask = appointents_df['store_name'].str.contains(unidade_regex_inclusiva, na=False, case=False) # Filtra somente unidades selecionadas

# Cria coluna flag de agendamento
appointents_df["eh_agendamento"] =  appointents_df["status_label"].str.contains(agendado_regex, na=False, case=False)

# Filtra base de agendamentos
filtered_df = appointents_df.loc[procedimento_mask & unidade_incluir_mask]

# Acrescenta coluna com a quantidade de mensagens por status (cancelado, falta)
filtered_df = filtered_df.merge(contar_mensagens, how='left', left_on=["telephones","status_label"], right_on=["telefone","status_fluxo"])
filtered_df.drop(columns=["telefone","status_fluxo"],inplace=True)
filtered_df.fillna(0,inplace=True)

# Adiciona coluna com a flag que diz se o cliente tem algum agendamento
filtered_df["tem agendamento"] = filtered_df.groupby(by=["telephones"])["eh_agendamento"].transform("max")

# Cria base telefones únicos para enviar mensagem
df_para_mandar_mensagens = filtered_df.groupby(by=["telephones","status_label"]).agg({"dias_do_agendamento":"min",'mensagens_count':'max',"store_name":"first","customer_name":"first", "employee_name":"first","startDate":"max", "hora_agendamento":"max"}).reset_index()

status_atendido_regex = r".*Atendido.*|.*Falta.*|.*Cancelado.*"
status_atendido_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_atendido_regex, na=False, case=False) # Remove Atendio e Confirmado
status_reagendado = 'Reagendado'
status_reagendado_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_reagendado, na=False, case=False) # Remove Reagendado

df_para_mandar_mensagens = df_para_mandar_mensagens.loc[status_atendido_mask & status_reagendado_mask]

# Pega telefones únicos com a data mais recente
df_para_mandar_mensagens = df_para_mandar_mensagens.loc[df_para_mandar_mensagens.groupby('telephones')['dias_do_agendamento'].idxmin()]

In [ ]:
# Final treatment
df_para_mandar_mensagens['mensagens_count'] = df_para_mandar_mensagens['mensagens_count'].astype(int)
df_para_mandar_mensagens['hora_agendamento'] = df_para_mandar_mensagens['hora_agendamento'].astype(str).str[:-3]
df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate,hora_agendamento
0,1144576148,Agendado,-2,0,HOMA,Juciara Souza Marques,Erik de Albuquerque,2024-11-14,13:15
1,1158419056,Agendado,-2,0,HOMA,Raquel Trindade Costa,Erik de Albuquerque,2024-11-14,12:15
4,11910604043,Agendado,-2,0,HOMA,Clarice Beatriz Alves dos Santos,Benedito Gilberto de Almeida Jr,2024-11-14,17:00
6,11911565595,Agendado,-2,0,HOMA,Tatiane Alexandra de Paula,Benedito Gilberto de Almeida Jr,2024-11-14,17:30
7,11911790503,Agendado,-1,1,HOMA,Glaucia Patrícia Nascimento,Jose Antônio Ferreira Brandão,2024-11-13,10:15
...,...,...,...,...,...,...,...,...,...
350,11998857909,Confirmado,0,1,HOMA,Elaine soares do Nascimento,Jose Antônio Ferreira Brandão,2024-11-12,10:15
355,12988376195,Agendado,-2,0,HOMA,Pamela Aparecida,Vanessa Contato Lopes Resende,2024-11-14,14:00
362,19989459859,Agendado,-1,1,HOMA,Poliana Marcelino Leonardo,Vanessa Contato Lopes Resende,2024-11-13,14:30
363,41996250658,Agendado,-2,0,HOMA,Iara Siqueira da Silva,Benedito Gilberto de Almeida Jr,2024-11-14,17:00


In [ ]:
df_para_mandar_mensagens.loc[df_para_mandar_mensagens["telephones"] == "11917197173"]

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate,hora_agendamento


In [ ]:

# # Create a sample DataFrame with fake data
# data = {
#     "telephones": ["11963546222"],
#     "status_label": ["Agendado"],
#     "dias_do_agendamento": [-2],
#     "mensagens_count": [0],
#     "store_name": ["HOMA"],
#     "customer_name": ["Luis"],
#     "employee_name": ["Dr X"],
#     "startDate": [datetime.today() - timedelta(days=x) for x in [1]],
#     "hora_agendamento": ["10:30"],
#     "email": ["lfariabr@gmail.com"]
# }

# # Convert to DataFrame
# df_para_mandar_mensagens = pd.DataFrame(data)
# df_para_mandar_mensagens

In [ ]:
# import pandas as pd
# import smtplib
# from email.mime.multipart import MIMEMultipart
# from email.mime.text import MIMEText
# from datetime import datetime
# from time import sleep

# # Function to send an invite via email
# def enviar_convite(para, subject, corpo):
#     # SMTP Configuration - You'll need to set this up with your SMTP credentials
#     smtp_server = 'smtp.gmail.com'
#     smtp_port = 587
#     username = 'rpdprocorpo@gmail.com'  # Replace with your Gmail address
#     password = 'rpdprocorpo@gmail.com@2021!@///*'     # Replace with your app-specific password

#     # Create the message
#     msg = MIMEMultipart()
#     msg['From'] = username
#     msg['To'] = para
#     msg['Subject'] = subject
#     msg.attach(MIMEText(corpo, 'plain'))

#     try:
#         # Establish connection to the SMTP server
#         server = smtplib.SMTP(smtp_server, smtp_port)
#         server.starttls()  # Upgrade the connection to a secure one using TLS
#         server.login(username, password)

#         # Send the email
#         server.sendmail(msg['From'], msg['To'], msg.as_string())
#         server.quit()

#         print(f"Convite enviado com sucesso para: {para}")
#         return True

#     except Exception as e:
#         print(f"Erro ao enviar para {para}: {str(e)}")
#         return False


# # Sending invites to contacts from df_para_mandar_mensagens
# data_de_envio = datetime.today().strftime('%Y-%m-%d')

# for index, row in df_para_mandar_mensagens.iterrows():
#     email = row["email"]  # Assuming your dataframe has an 'email' column for contacts
#     status_label = row["status_label"]
#     mensagens_count = row["mensagens_count"]
#     dias_do_agendamento = row["dias_do_agendamento"]

#     # Agendado d-2
#     if status_label == "Agendado" and dias_do_agendamento == -2 and mensagens_count == 0:
#         subject = "Convite para Agendamento"
#         mensagem_texto = "Você está convidado a participar do nosso agendamento conforme combinado. Favor confirmar sua presença."

#         success = enviar_convite(email, subject, mensagem_texto)

#         if success:
#             # Assuming a function to update csv after sending messages successfully
#             update_csv_mensagens([email, status_label, "convite", data_de_envio])
#         else:
#             print(f"Erro ao enviar convite para: {email}")

#         sleep(10)
#         print("Descansando por 10 segundos...")

### Send Messages

In [ ]:
from datetime import datetime

def ajusta_mensagem(row, texto_mensagem):
    store_name = row["store_name"].capitalize()
    customer_name = row["customer_name"].split()[0].capitalize()
    prestador_name = row["employee_name"].split()[0].capitalize()
    hora = row["hora_agendamento"]

    # Convert startDate to the desired format if it's a Timestamp or datetime object
    startDate = row["startDate"]
    if isinstance(startDate, datetime):
        startDate = startDate.strftime('%d/%m/%Y')
    else:
        startDate = datetime.strptime(str(startDate), '%Y-%m-%d').strftime('%d/%m/%Y')

    texto_mensagem = texto_mensagem.replace("[nome]", customer_name)
    texto_mensagem = texto_mensagem.replace("[prestador]", prestador_name)
    texto_mensagem = texto_mensagem.replace("[data]", startDate)
    texto_mensagem = texto_mensagem.replace("[hora]", hora)

    return texto_mensagem

In [ ]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-4
  if status_label == "Agendado" and dias_do_agendamento == -4:
    nome_mensagem = "agd_d4"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]
    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)
    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Agendado d-1
  if status_label == "Agendado" and dias_do_agendamento == -1:
    nome_mensagem = "agd_d1"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-1
  if status_label == "Confirmado" and dias_do_agendamento == -1:
    nome_mensagem = "conf_d1"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-0
  if status_label == "Confirmado" and dias_do_agendamento == 0:
    nome_mensagem = "conf_d0"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-0
  if status_label == "Agendado" and dias_do_agendamento == 0:
    nome_mensagem = "agd_d0"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

agd_d1 enviada com sucesso para: 11911790503
Resting 10s...
conf_d1 enviada com sucesso para: 11912697560
Resting 10s...
conf_d1 enviada com sucesso para: 11913050738
Resting 10s...
conf_d0 enviada com sucesso para: 11915806358
Resting 10s...
agd_d1 enviada com sucesso para: 11916162747
Resting 10s...
agd_d1 enviada com sucesso para: 1192283981
Resting 10s...
conf_d1 enviada com sucesso para: 11932113736
Resting 10s...
agd_d1 enviada com sucesso para: 11937452980
Resting 10s...
conf_d0 enviada com sucesso para: 11937747681
Resting 10s...
conf_d1 enviada com sucesso para: 11939445775
Resting 10s...
conf_d0 enviada com sucesso para: 11940363015
Resting 10s...
agd_d1 enviada com sucesso para: 11941591032
Resting 10s...
agd_d0 enviada com sucesso para: 11943725790
Resting 10s...
conf_d1 enviada com sucesso para: 11944664529
Resting 10s...
conf_d1 enviada com sucesso para: 11944955243
Resting 10s...
agd_d0 enviada com sucesso para: 11945243450
Resting 10s...
agd_d0 enviada com sucesso para:

# Pró-Corpo

### SocialHub

In [6]:
# Envias as mensagens por meio da API do Social Hub
# 1144503626: UT1pYNyMuth8XfBIrxSq41hb95Vhof8n

url = "https://apinew.socialhub.pro/api/sendMessage"
api_token = 'UT1pYNyMuth8XfBIrxSq41hb95Vhof8n'

def send_message(telephone, message):

    telephone = str(telephone)
    request_data = {
        "api_token": api_token,  # ID da API no Social HUB BOTOX
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    response = requests.post(url, headers=headers, data=json.dumps(request_data))

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def send_message_with_file(telephone, message,file_path):
    url = "https://apinew.socialhub.pro/api/sendMessage"

    telephone = str(telephone)

    request_data = {
        "api_token": api_token,
        "phone": telephone,
        "message": message,
        "preview_url": True
    }

    headers = {
        "Content-Type": "application/json"
    }

    with open(file_path, 'rb') as file:
        files = {'file': file}
        response = requests.post(url, data=request_data, files=files)

    if response.status_code != 200:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")
        return None

    data = response.json()

    return data

def enviar_mensagem(telefone,mensagem,file_name):

  if file_name == None:
    data = send_message(telefone,mensagem)
  else:
    data = send_message_with_file(telefone,mensagem,file_name)

  return data

### Load DBs

In [7]:
# Função para atualizar CSV de Mensagens

def update_csv_mensagens_hsr(dados_mensagens):
  with open(mensagens_log_file_procorpo, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile, delimiter=',')
    writer.writerow(dados_mensagens)

In [8]:
# Carregando bases

appointents_file = "/content/drive/MyDrive/_whatsapp-project/appointments_30d/appointments_5d.csv"
mensagens_log_file_procorpo = "/content/drive/MyDrive/_whatsapp-project/sent_messages_confirmation_procorpo.csv"

appointents_df = pd.read_csv(appointents_file) # Relatório de agendamentos do CRM
base_mensagens_df_procorpo = pd.read_csv(mensagens_log_file_procorpo) # Log de mensagens enviadas

In [9]:
import re

# Função para limpar telefones
def clean_telephone(telefone):
    telefone = str(telefone).split('.')[0]  # Remove qualquer coisa após o ponto decimal
    telefone = re.sub(r'[^\d]', '', telefone)  # Remove todos os caracteres não numéricos
    if telefone.startswith('55'):
        telefone = telefone[2:]  # Remove o prefixo +55
    return telefone

# Arruma os telefones
appointents_df["telephones"] = appointents_df["telephones"].astype(str)
appointents_df["telephones"] = appointents_df["telephones"].apply(clean_telephone)

In [10]:
# create new column data_de_envio
base_mensagens_df_procorpo["data_de_envio"] = datetime.today().strftime('%Y-%m-%d')
# Filtra log de mensagens excluindo mensagens enviadas há mais de 30 dias
base_mensagens_df_procorpo['data_de_envio'] = pd.to_datetime(base_mensagens_df_procorpo['data_de_envio'])
date_30_days_ago = datetime.now() - timedelta(days=30)
base_mensagens_df_procorpo = base_mensagens_df_procorpo[base_mensagens_df_procorpo['data_de_envio'] >= date_30_days_ago]

# Groupby do log de mensagens
contar_mensagens = base_mensagens_df_procorpo.groupby(["telefone","status_fluxo"]).agg({"telefone":"count"})
contar_mensagens = contar_mensagens.rename(columns={"telefone": "mensagens_count"})

contar_mensagens = contar_mensagens.reset_index()
contar_mensagens["telefone"] = contar_mensagens["telefone"].astype(str)

In [11]:
# Pega o texto e outros dados das mensagens

url_da_tabela_de_mensagens = 'https://docs.google.com/spreadsheets/d/1iNt8-2qmQKYD0McDDrUUb9QLBn5G2h6M5yUa8oo2isg/edit'
ss = gc.open_by_url(url_da_tabela_de_mensagens)
sheet = ss.worksheet("Confirma_PRO")
mensagens = sheet.get_all_records()

# Transforma os dados em um dicionário para hash indexing
mensagens_dic = {}
for item in mensagens:
  tipo_de_mensagem = item["nome_mensagem"]
  mensagens_dic[tipo_de_mensagem] = item

In [12]:
mensagens_dic

{'agd_d2_11am': {'nome_mensagem': 'agd_d2_11am',
  'texto_mensagem': 'Olá, tudo bem?\n\nAqui é Isa da Pró-Corpo Estética [unidade]! 💎\n\nPassando aqui para lembrar da sua avaliação.\nEstamos felizes que você decidiu dar esse passo super importante para sua autoestima. 🤩\nPodemos confirmar sua presença?\n\nNão se preocupe que iremos entrar em contato novamente para te lembrar e passar mais informações.\nSe precisar reagendar é só mandar um “Alô” por aqui, conte comigo. 😃\nBjs.',
  'file_id': '',
  'descrição_mensagem': ''},
 'agd_d1_03pm': {'nome_mensagem': 'agd_d1_03pm',
  'texto_mensagem': 'Oiee...\n\nPassando aqui para te lembrar que o grande dia é amanhã. 💃\nNossa profissional estará a sua espera as [hora].\n📍 Estamos localizados na [address]\n\nAh, e falando em coisa boa sua avaliação é gratuita, mas é por tempo limitado. 😁\nAproveite.',
  'file_id': '',
  'descrição_mensagem': ''},
 'agd_d1_05pm': {'nome_mensagem': 'agd_d1_05pm',
  'texto_mensagem': 'Oiee.. tudo bem?\n\nPosso conf

### Data Prep

In [13]:
# Converte a coluna 'startDate' para o formato datetime
appointents_df['startDate'] = pd.to_datetime(appointents_df['startDate'])

# Cria uma nova coluna 'hora_agendamento' com apenas a hora do agendamento
appointents_df['hora_agendamento'] = appointents_df['startDate'].dt.time

# Normaliza a data (remove a hora da coluna 'startDate')
appointents_df['startDate'] = appointents_df['startDate'].dt.normalize()

# Calcula a diferença de dias
today = datetime.today()
appointents_df['dias_do_agendamento'] = (today - appointents_df['startDate']).dt.days

In [14]:
agendado_regex = r".*Agendado.*|.*Reagendado.*|.*Atendido.*"
unidade_regex_inclusiva = "JARDINS" # Adicionar unidades que queremos ver no relatório

# Lista de procedimentos a serem excluídos
procedimentos_inclusao = [
    'AVALIAÇÃO INJETÁVEIS E INVASIVOS',
    'AVALIAÇÃO ESTÉTICA'
]

# Atualiza a máscara para filtrar procedimentos válidos excluindo os indesejados
procedimento_mask = appointents_df['procedure_name'].isin(procedimentos_inclusao) # Exclui procedimentos indesejados
unidade_incluir_mask = appointents_df['store_name'].str.contains(unidade_regex_inclusiva, na=False, case=False) # Filtra somente unidades selecionadas

# Cria coluna flag de agendamento
appointents_df["eh_agendamento"] =  appointents_df["status_label"].str.contains(agendado_regex, na=False, case=False)

# Filtra base de agendamentos
filtered_df = appointents_df.loc[procedimento_mask & unidade_incluir_mask]

# Acrescenta coluna com a quantidade de mensagens por status (cancelado, falta)
filtered_df = filtered_df.merge(contar_mensagens, how='left', left_on=["telephones","status_label"], right_on=["telefone","status_fluxo"])
filtered_df.drop(columns=["telefone","status_fluxo"],inplace=True)
filtered_df.fillna(0,inplace=True)

# Adiciona coluna com a flag que diz se o cliente tem algum agendamento
filtered_df["tem agendamento"] = filtered_df.groupby(by=["telephones"])["eh_agendamento"].transform("max")

# Cria base telefones únicos para enviar mensagem
df_para_mandar_mensagens = filtered_df.groupby(by=["telephones","status_label"]).agg({"dias_do_agendamento":"min",'mensagens_count':'max',"store_name":"first","customer_name":"first", "employee_name":"first","startDate":"max", "hora_agendamento":"max"}).reset_index()

status_atendido_regex = r".*Atendido.*|.*Falta.*|.*Cancelado.*"
status_atendido_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_atendido_regex, na=False, case=False) # Remove Atendio e Confirmado
status_reagendado = 'Reagendado'
status_reagendado_mask = ~df_para_mandar_mensagens['status_label'].str.contains(status_reagendado, na=False, case=False) # Remove Reagendado

df_para_mandar_mensagens = df_para_mandar_mensagens.loc[status_atendido_mask & status_reagendado_mask]

# Pega telefones únicos com a data mais recente
df_para_mandar_mensagens = df_para_mandar_mensagens.loc[df_para_mandar_mensagens.groupby('telephones')['dias_do_agendamento'].idxmin()]

In [15]:
# Final treatment
df_para_mandar_mensagens['mensagens_count'] = df_para_mandar_mensagens['mensagens_count'].astype(int)
df_para_mandar_mensagens['hora_agendamento'] = df_para_mandar_mensagens['hora_agendamento'].astype(str).str[:-3]
df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate,hora_agendamento
0,11440364964,Agendado,-2,0,JARDINS,Maria Valéria Silva De Medina,Cintia dos Reis,2024-11-23,10:00
2,11912181372,Agendado,-2,0,JARDINS,Francisco Antônio Pereira de Melo Júnior,Avaliação Jardins,2024-11-23,09:00
3,11912959691,Agendado,-1,0,JARDINS,Isabelle Domingos Barbosa,Cintia dos Reis,2024-11-22,19:15
5,11913180351,Agendado,-2,0,JARDINS,Meire Marega Rodrigues,Cintia dos Reis,2024-11-23,12:45
6,11913577974,Agendado,-4,0,JARDINS,Lili Ventorin,Cintia dos Reis,2024-11-25,17:45
...,...,...,...,...,...,...,...,...,...
198,17992100495,Agendado,0,1,JARDINS,Cleber Hermenegildo Rossi,Avaliação Jardins,2024-11-21,09:45
199,19983514764,Confirmado,0,0,JARDINS,Beatriz Caroline Reis Justino,Avaliação Jardins,2024-11-21,11:00
201,19999295395,Agendado,-2,0,JARDINS,Caroline Salzati,Cintia dos Reis,2024-11-23,13:15
203,24981280088,Agendado,-5,0,JARDINS,Nanci Elizabete,Cintia dos Reis,2024-11-26,20:15


In [16]:

# # Create a sample DataFrame with fake data
# data = {
#     "telephones": ["11963546222"],
#     "status_label": ["Agendado"],
#     "dias_do_agendamento": [-1],
#     "mensagens_count": [0],
#     "store_name": ["JARDINS",],
#     "customer_name": ["Luis"],
#     "employee_name": ["Dr X"],
#     "startDate": [datetime.today() - timedelta(days=x) for x in [1]],
#     "hora_agendamento": ["10:30"]
# }

# # Convert to DataFrame
# df_para_mandar_mensagens = pd.DataFrame(data)
# df_para_mandar_mensagens

In [17]:
address_list = {
    'JARDINS': "R. Augusta, 2677 - Cerqueira César, São Paulo"
}

# address_list to dataframe
df_address_list = pd.DataFrame(list(address_list.items()), columns=['store_name', 'address'])
df_address_list

,store_name,address
0,JARDINS,"R. Augusta, 2677 - Cerqueira César, São Paulo"


In [18]:
# Concat df_para_mandar_mensagens and df_address_list at "store_name"
df_para_mandar_mensagens = pd.merge(df_para_mandar_mensagens, df_address_list, on='store_name', how='left')
df_para_mandar_mensagens

,telephones,status_label,dias_do_agendamento,mensagens_count,store_name,customer_name,employee_name,startDate,hora_agendamento,address
0,11440364964,Agendado,-2,0,JARDINS,Maria Valéria Silva De Medina,Cintia dos Reis,2024-11-23,10:00,"R. Augusta, 2677 - Cerqueira César, São Paulo"
1,11912181372,Agendado,-2,0,JARDINS,Francisco Antônio Pereira de Melo Júnior,Avaliação Jardins,2024-11-23,09:00,"R. Augusta, 2677 - Cerqueira César, São Paulo"
2,11912959691,Agendado,-1,0,JARDINS,Isabelle Domingos Barbosa,Cintia dos Reis,2024-11-22,19:15,"R. Augusta, 2677 - Cerqueira César, São Paulo"
3,11913180351,Agendado,-2,0,JARDINS,Meire Marega Rodrigues,Cintia dos Reis,2024-11-23,12:45,"R. Augusta, 2677 - Cerqueira César, São Paulo"
4,11913577974,Agendado,-4,0,JARDINS,Lili Ventorin,Cintia dos Reis,2024-11-25,17:45,"R. Augusta, 2677 - Cerqueira César, São Paulo"
...,...,...,...,...,...,...,...,...,...,...
90,17992100495,Agendado,0,1,JARDINS,Cleber Hermenegildo Rossi,Avaliação Jardins,2024-11-21,09:45,"R. Augusta, 2677 - Cerqueira César, São Paulo"
91,19983514764,Confirmado,0,0,JARDINS,Beatriz Caroline Reis Justino,Avaliação Jardins,2024-11-21,11:00,"R. Augusta, 2677 - Cerqueira César, São Paulo"
92,19999295395,Agendado,-2,0,JARDINS,Caroline Salzati,Cintia dos Reis,2024-11-23,13:15,"R. Augusta, 2677 - Cerqueira César, São Paulo"
93,24981280088,Agendado,-5,0,JARDINS,Nanci Elizabete,Cintia dos Reis,2024-11-26,20:15,"R. Augusta, 2677 - Cerqueira César, São Paulo"


### Send Messages

In [19]:
from datetime import datetime

def ajusta_mensagem(row, texto_mensagem):
    store_name = row["store_name"].capitalize()
    customer_name = row["customer_name"].split()[0].capitalize()
    prestador_name = row["employee_name"].split()[0].capitalize()
    store_name = row["store_name"].capitalize()
    hora = row["hora_agendamento"]
    address = row["address"]

    # Convert startDate to the desired format if it's a Timestamp or datetime object
    startDate = row["startDate"]
    if isinstance(startDate, datetime):
        startDate = startDate.strftime('%d/%m/%Y')
    else:
        startDate = datetime.strptime(str(startDate), '%Y-%m-%d').strftime('%d/%m/%Y')

    texto_mensagem = texto_mensagem.replace("[nome]", customer_name)
    texto_mensagem = texto_mensagem.replace("[prestador]", prestador_name)
    texto_mensagem = texto_mensagem.replace("[data]", startDate)
    texto_mensagem = texto_mensagem.replace("[hora]", hora)
    texto_mensagem = texto_mensagem.replace("[unidade]", store_name)
    texto_mensagem = texto_mensagem.replace("[address]", address)

    return texto_mensagem

#### Agd/Conf d0-09:00

In [20]:
import time

# Total time to sleep
total_time = 1 * 90 * 45  #+ (15 * 60) # 1 hour + 15m
interval = 60
display_interval = 1  # countdown
# Loop that sleeps until time to go
for remaining_time in range(int(total_time), 0, -interval):

    time.sleep(interval)
    if remaining_time % (display_interval * 60) == 0:
        remaining_minutes = remaining_time // 60
        print(f"Time left: {remaining_minutes} minutes")

print("Finished waiting.. ready for next")

Finished waiting.. ready for next


In [21]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-0
  if status_label == "Agendado" and dias_do_agendamento == 0:# and mensagens_count == 0:
    nome_mensagem = "agd_d0_09am"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

  # Confirmado d-0
  if status_label == "Confirmado" and dias_do_agendamento == 0:# and mensagens_count == 0:
    nome_mensagem = "conf_d0_09am"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

agd_d0_09am enviada com sucesso para: 11916543052
Resting 10s...
agd_d0_09am enviada com sucesso para: 11932337444
Resting 10s...
agd_d0_09am enviada com sucesso para: 11949745065
Resting 10s...
agd_d0_09am enviada com sucesso para: 11952919316
Resting 10s...
agd_d0_09am enviada com sucesso para: 11954229059
Resting 10s...
agd_d0_09am enviada com sucesso para: 11958670411
Resting 10s...
conf_d0_09am enviada com sucesso para: 11959038235
Resting 10s...
conf_d0_09am enviada com sucesso para: 11964275538
Resting 10s...
agd_d0_09am enviada com sucesso para: 11964346712
Resting 10s...
agd_d0_09am enviada com sucesso para: 11964785555
Resting 10s...
agd_d0_09am enviada com sucesso para: 11970393854
Resting 10s...
agd_d0_09am enviada com sucesso para: 11973195404
Resting 10s...
conf_d0_09am enviada com sucesso para: 11974421266
Resting 10s...
agd_d0_09am enviada com sucesso para: 11976247171
Resting 10s...
agd_d0_09am enviada com sucesso para: 11978918180
Resting 10s...
agd_d0_09am enviada co

#### Agd d2-12:00

In [22]:
import time

# Total time to sleep
total_time = 1 * 60 * 115  #+ (15 * 60) # 1 hour + 15m
interval = 60
display_interval = 1  # countdown
# Loop that sleeps until time to go
for remaining_time in range(int(total_time), 0, -interval):

    time.sleep(interval)
    if remaining_time % (display_interval * 60) == 0:
        remaining_minutes = remaining_time // 60
        print(f"Time left: {remaining_minutes} minutes")

print("Finished waiting.. ready for next")

Time left: 115 minutes
Time left: 114 minutes
Time left: 113 minutes
Time left: 112 minutes
Time left: 111 minutes
Time left: 110 minutes
Time left: 109 minutes
Time left: 108 minutes
Time left: 107 minutes
Time left: 106 minutes
Time left: 105 minutes
Time left: 104 minutes
Time left: 103 minutes
Time left: 102 minutes
Time left: 101 minutes
Time left: 100 minutes
Time left: 99 minutes
Time left: 98 minutes
Time left: 97 minutes
Time left: 96 minutes
Time left: 95 minutes
Time left: 94 minutes
Time left: 93 minutes
Time left: 92 minutes
Time left: 91 minutes
Time left: 90 minutes
Time left: 89 minutes
Time left: 88 minutes
Time left: 87 minutes
Time left: 86 minutes
Time left: 85 minutes
Time left: 84 minutes
Time left: 83 minutes
Time left: 82 minutes
Time left: 81 minutes
Time left: 80 minutes
Time left: 79 minutes
Time left: 78 minutes
Time left: 77 minutes
Time left: 76 minutes
Time left: 75 minutes
Time left: 74 minutes
Time left: 73 minutes
Time left: 72 minutes
Time left: 71 mi

In [23]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-2
  if status_label == "Agendado" and dias_do_agendamento == -2:# and mensagens_count == 0:
    nome_mensagem = "agd_d2_11am"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

agd_d2_11am enviada com sucesso para: 11440364964
Resting 10s...
agd_d2_11am enviada com sucesso para: 11912181372
Resting 10s...
agd_d2_11am enviada com sucesso para: 11913180351
Resting 10s...
agd_d2_11am enviada com sucesso para: 11932068256
Resting 10s...
agd_d2_11am enviada com sucesso para: 11942440339
Resting 10s...
agd_d2_11am enviada com sucesso para: 11947119333
Resting 10s...
agd_d2_11am enviada com sucesso para: 11949190936
Resting 10s...
agd_d2_11am enviada com sucesso para: 11952274542
Resting 10s...
agd_d2_11am enviada com sucesso para: 11952402798
Resting 10s...
agd_d2_11am enviada com sucesso para: 11959018706
Resting 10s...
agd_d2_11am enviada com sucesso para: 11968621411
Resting 10s...
agd_d2_11am enviada com sucesso para: 11970977364
Resting 10s...
agd_d2_11am enviada com sucesso para: 11983040321
Resting 10s...
agd_d2_11am enviada com sucesso para: 11983478912
Resting 10s...
agd_d2_11am enviada com sucesso para: 11993601010
Resting 10s...
agd_d2_11am enviada com s

#### Agd d1-15:00


In [24]:
import time

# Total time to sleep
total_time = 1 * 60 * 150  #+ (15 * 60) # 1 hour + 15m
interval = 60
display_interval = 1  # countdown
# Loop that sleeps until time to go
for remaining_time in range(int(total_time), 0, -interval):

    time.sleep(interval)
    if remaining_time % (display_interval * 60) == 0:
        remaining_minutes = remaining_time // 60
        print(f"Time left: {remaining_minutes} minutes")

print("Finished waiting.. ready for next")

Time left: 150 minutes
Time left: 149 minutes
Time left: 148 minutes
Time left: 147 minutes
Time left: 146 minutes
Time left: 145 minutes
Time left: 144 minutes
Time left: 143 minutes
Time left: 142 minutes
Time left: 141 minutes
Time left: 140 minutes
Time left: 139 minutes
Time left: 138 minutes
Time left: 137 minutes
Time left: 136 minutes
Time left: 135 minutes
Time left: 134 minutes
Time left: 133 minutes
Time left: 132 minutes
Time left: 131 minutes
Time left: 130 minutes
Time left: 129 minutes
Time left: 128 minutes
Time left: 127 minutes
Time left: 126 minutes
Time left: 125 minutes
Time left: 124 minutes
Time left: 123 minutes
Time left: 122 minutes
Time left: 121 minutes
Time left: 120 minutes
Time left: 119 minutes
Time left: 118 minutes
Time left: 117 minutes
Time left: 116 minutes
Time left: 115 minutes
Time left: 114 minutes
Time left: 113 minutes
Time left: 112 minutes
Time left: 111 minutes
Time left: 110 minutes
Time left: 109 minutes
Time left: 108 minutes
Time left: 

In [25]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-1
  if status_label == "Agendado" and dias_do_agendamento == -1:# and mensagens_count == 0:
    nome_mensagem = "agd_d1_03pm"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")

agd_d1_03pm enviada com sucesso para: 11912959691
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11934040329
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11940604215
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11947048091
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11952510570
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11959084401
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11968101680
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11970567798
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11971695102
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11973304667
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11975897078
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11976953948
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11984021862
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11986901585
Resting 10s...
agd_d1_03pm enviada com sucesso para: 11993636272
Resting 10s...
agd_d1_03pm enviada com s

#### Agd d1-17:00

In [ ]:
import time

# Total time to sleep
total_time = 1 * 60 * 110  #+ (15 * 60) # 1 hour + 15m
interval = 60
display_interval = 1  # countdown
# Loop that sleeps until time to go
for remaining_time in range(int(total_time), 0, -interval):

    time.sleep(interval)
    if remaining_time % (display_interval * 60) == 0:
        remaining_minutes = remaining_time // 60
        print(f"Time left: {remaining_minutes} minutes")

print("Finished waiting.. ready for next")

Time left: 110 minutes
Time left: 109 minutes
Time left: 108 minutes
Time left: 107 minutes
Time left: 106 minutes
Time left: 105 minutes
Time left: 104 minutes
Time left: 103 minutes
Time left: 102 minutes
Time left: 101 minutes
Time left: 100 minutes
Time left: 99 minutes
Time left: 98 minutes
Time left: 97 minutes
Time left: 96 minutes
Time left: 95 minutes
Time left: 94 minutes
Time left: 93 minutes
Time left: 92 minutes
Time left: 91 minutes
Time left: 90 minutes
Time left: 89 minutes
Time left: 88 minutes
Time left: 87 minutes
Time left: 86 minutes
Time left: 85 minutes
Time left: 84 minutes
Time left: 83 minutes
Time left: 82 minutes
Time left: 81 minutes
Time left: 80 minutes
Time left: 79 minutes
Time left: 78 minutes
Time left: 77 minutes
Time left: 76 minutes
Time left: 75 minutes
Time left: 74 minutes
Time left: 73 minutes


In [ ]:
import time

data_de_envio = datetime.today().strftime('%Y-%m-%d')

for index, row in df_para_mandar_mensagens.iterrows():

  telephone = row["telephones"]
  status_label = row["status_label"]
  mensagens_count = row["mensagens_count"]
  dias_do_agendamento = row["dias_do_agendamento"]

  # Agendado d-1
  if status_label == "Agendado" and dias_do_agendamento == -1:# and mensagens_count == 0:
    nome_mensagem = "agd_d1_05pm"
    mensagem = mensagens_dic[nome_mensagem]
    texto_mensagem = ajusta_mensagem(row,mensagem["texto_mensagem"])
    file_id = mensagem["file_id"]

    if file_id == '':
      file_id = None
    response = enviar_mensagem(telephone, texto_mensagem, file_id)

    if response["success"] == True:
        update_csv_mensagens_hsr([telephone, status_label, nome_mensagem, data_de_envio])
        print(f"{nome_mensagem} enviada com sucesso para: {telephone}")
    else:
        print(f"erro {response}")
    sleep(10)
    print("Resting 10s...")